# Türkçe MMLU Benchmark Testi (Hızlandırılmış Batch Inference)

Bu notebook, fine-tune edilen **`gururaser/qwen3-4b-bilimkurgu`** ve taban model **`unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit`** modellerini **Batch Inference (Batch Size=16/32)** ve **Lazy SentenceTransformer** optimizasyonları ile **8-10 kat daha hızlı** Türkçe MMLU testine sokar.

In [1]:
# 1. Gerekli kütüphanelerin kurulması
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers datasets pandas huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 126.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 140.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<

In [2]:
# 2. Bağımlılıklar ve Optimize Edilmiş Anlamsal Karşılaştırma Mantığı
import time
import torch
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import HfApi, login

print("Anlamsal benzerlik modeli hazırlanıyor...")
anlamsal_benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

def cevap_dogru_mu(dogru_cevap_index, verilen_cevap, secenekler):
    harfler = ['A', 'B', 'C', 'D', 'E']
    dogru_harf = harfler[dogru_cevap_index]
    verilen_cevap = verilen_cevap.upper().strip()

    # 1. Hızlı Harf Kontrolü (A, B, C...)
    if dogru_harf == verilen_cevap:
        return True
    elif len(verilen_cevap) > 1 and verilen_cevap[1] in [" ", ":", ")", "=", "-", "."]:
        return dogru_harf == verilen_cevap[0]
    elif len(verilen_cevap) > 0 and verilen_cevap[0] in harfler and verilen_cevap[0] == dogru_harf:
        return True
    else:
        # 2. Sadece harf tutmazsa Anlamsal Vektör Karşılaştırması Yap (Lazy Evaluation)
        try:
            encoded_cevap = anlamsal_benzerlik_modeli.encode([verilen_cevap])
            encoded_secenekler = anlamsal_benzerlik_modeli.encode(secenekler)
            benzerlik_listesi = anlamsal_benzerlik_modeli.similarity(encoded_cevap, encoded_secenekler).tolist()[0]
            en_yuksek_benzerlik_index = benzerlik_listesi.index(max(benzerlik_listesi))
            return en_yuksek_benzerlik_index == dogru_cevap_index
        except:
            return False

Anlamsal benzerlik modeli hazırlanıyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# 3. Hızlandırılmış Batch Inference Test Fonksiyonu
def benchmark_model_batched(model_name: str, mmlu_veri: pd.DataFrame, batch_size: int = 16):
    print(f"\n=========================================")
    print(f"Model Test Ediliyor (Batch Size={batch_size}): {model_name}")
    print(f"=========================================")
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    dogru_cevap_sayisi = 0
    model_bolum_sonuc = {}
    baslama_zamani = time.time()
    toplam_soru = len(mmlu_veri)
    
    prompts = []
    for i in range(toplam_soru):
        soru_text = mmlu_veri.iloc[i]['soru'] + "\n"
        harfler = ['A', 'B', 'C', 'D', 'E']
        secenekler = mmlu_veri.iloc[i]['secenekler']
        for j in range(len(secenekler)):
            soru_text += f"{harfler[j]}: {secenekler[j]}\n"

        prompt = (
            "Sana soru ve seçenekleri veriyorum. sadece hangi seçeneğin sorunun doğru cevabı olduğunu yaz. "
            "Örneğin 'A' veya 'B' gibi. Lütfen herhangi bir açıklama yapma!\nSoru: " + soru_text
        )
        prompts.append(prompt)

    with torch.inference_mode():
        for b_idx in range(0, toplam_soru, batch_size):
            batch_prompts = prompts[b_idx:b_idx + batch_size]
            batch_data = mmlu_veri.iloc[b_idx:b_idx + batch_size]
            
            inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
            outputs = model.generate(
                **inputs,
                max_new_tokens=3, # Sadece harf çıktısı için token kısıtlaması (Hız kazandırır)
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            
            input_len = inputs.input_ids.shape[1]
            generated_tokens = outputs[:, input_len:]
            responses = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            
            for row_idx, cevap in enumerate(responses):
                cevap_clean = cevap.strip()
                secenekler = batch_data.iloc[row_idx]['secenekler']
                dogru_cevap_idx = batch_data.iloc[row_idx]['cevap']
                bolum = batch_data.iloc[row_idx]['bolum']
                
                if bolum not in model_bolum_sonuc:
                    model_bolum_sonuc[bolum] = {'dogru': 0, 'toplam': 0}
                model_bolum_sonuc[bolum]['toplam'] += 1
                
                if cevap_dogru_mu(dogru_cevap_idx, cevap_clean, secenekler):
                    dogru_cevap_sayisi += 1
                    model_bolum_sonuc[bolum]['dogru'] += 1

            current_count = min(b_idx + batch_size, toplam_soru)
            gecen = round(time.time() - baslama_zamani, 1)
            anlik_basari = round((dogru_cevap_sayisi / current_count) * 100, 2)
            hiz_sps = round(current_count / gecen, 1)
            print(f"\r[Soru {current_count}/{toplam_soru}] Doğru: {dogru_cevap_sayisi} | Başarı: %{anlik_basari} | Hız: {hiz_sps} soru/sn | Süre: {gecen}s", end="")

    gecen_sure = round(time.time() - baslama_zamani, 2)
    genel_basari = round((dogru_cevap_sayisi / toplam_soru) * 100, 2)
    print(f"\n\n--> {model_name} Testi Tamamlandı! Süre: {gecen_sure}s | Genel Başarı: %{genel_basari}")

    del model
    del tokenizer
    torch.cuda.empty_cache()

    return {
        "model": model_name,
        "genel_basari": genel_basari,
        "dogru_sayisi": dogru_cevap_sayisi,
        "toplam_soru": toplam_soru,
        "sure_sn": gecen_sure,
        "bolum_skorlari": model_bolum_sonuc
    }

In [4]:
# 4. Veri Setinin Yüklenmesi ve Testlerin Çalıştırılması (Batch Size = 16)
print("Türkçe MMLU Veri Seti İndiriliyor...")
mmlu_veri = pd.read_parquet("hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet")
print(f"Toplam MMLU Soru Sayısı: {len(mmlu_veri)}")

modeller = [
    "unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit", # Base Model
    "gururaser/qwen3-4b-bilimkurgu"                   # Fine-Tuned Model
]

sonuclar = []
for model_name in modeller:
    res = benchmark_model_batched(model_name, mmlu_veri, batch_size=64)
    sonuclar.append(res)

Türkçe MMLU Veri Seti İndiriliyor...
Toplam MMLU Soru Sayısı: 6200

Model Test Ediliyor (Batch Size=64): unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit


config.json:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.65k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[Soru 6016/6200] Doğru: 2075 | Başarı: %34.49 | Hız: 24.7 soru/sn | Süre: 243.2s

[transformers] Both `max_new_tokens` (=3) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Soru 6080/6200] Doğru: 2091 | Başarı: %34.39 | Hız: 24.7 soru/sn | Süre: 246.4s

[transformers] Both `max_new_tokens` (=3) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Soru 6200/6200] Doğru: 2130 | Başarı: %34.35 | Hız: 24.7 soru/sn | Süre: 251.2s

--> gururaser/qwen3-4b-bilimkurgu Testi Tamamlandı! Süre: 251.23s | Genel Başarı: %34.35


In [5]:
# 5. Karşılaştırmalı Tablo Oluşturulması
df_sonuclar = pd.DataFrame(sonuclar)
print("\n=================== BENCHMARK KARŞILAŞTIRMA TABLOSU ===================")
print(df_sonuclar[['model', 'genel_basari', 'dogru_sayisi', 'toplam_soru', 'sure_sn']])

markdown_table = "### 📊 Türkçe MMLU Benchmark Karşılaştırma Tablosu\n\n"
markdown_table += "| Model Türü | Model Adı | Genel Başarı (%) | Doğru / Toplam Soru | Test Süresi (sn) |\n"
markdown_table += "|---|---|---|---|---|\n"

for r in sonuclar:
    model_turu = "Base Model" if "unsloth" in r['model'] else "Fine-Tuned Model"
    markdown_table += f"| {model_turu} | `{r['model']}` | **%{r['genel_basari']}** | {r['dogru_sayisi']} / {r['toplam_soru']} | {r['sure_sn']}s |\n"

print("\nHugging Face README için Hazırlanan Markdown:")
print(markdown_table)


=================== BENCHMARK KARŞILAŞTIRMA TABLOSU ===================
                                             model  genel_basari  \
0  unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit         26.21   
1                    gururaser/qwen3-4b-bilimkurgu         34.35   

   dogru_sayisi  toplam_soru  sure_sn  
0          1625         6200   218.40  
1          2130         6200   251.23  

Hugging Face README için Hazırlanan Markdown:
### 📊 Türkçe MMLU Benchmark Karşılaştırma Tablosu

| Model Türü | Model Adı | Genel Başarı (%) | Doğru / Toplam Soru | Test Süresi (sn) |
|---|---|---|---|---|
| Base Model | `unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit` | **%26.21** | 1625 / 6200 | 218.4s |
| Fine-Tuned Model | `gururaser/qwen3-4b-bilimkurgu` | **%34.35** | 2130 / 6200 | 251.23s |



In [6]:
# 6. Hugging Face Model Kartının (README.md) Otomatik Güncellenmesi
import os
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = input("Lütfen Hugging Face Write Token girin: ")

login(token=HF_TOKEN)

repo_id = "gururaser/qwen3-4b-bilimkurgu"

guncel_readme = f"""---
base_model: unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit
tags:
- text-generation-inference
- transformers
- unsloth
- qwen3
- mmlu
- evaluation
license: cc-by-nc-4.0
language:
- tr
datasets:
- gururaser/ithaki-bilimkurgu-klasikleri
- alibayram/yapay_zeka_turkce_mmlu_model_cevaplari
---

# Qwen3 4B Bilimkurgu - Türkçe MMLU Benchmark & Model Kartı

- **Geliştirici:** [gururaser](https://huggingface.co/gururaser)
- **Lisans:** cc-by-nc-4.0
- **Eğitildiği Taban Model:** unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit
- **Eğitim Veri Seti:** [gururaser/ithaki-bilimkurgu-klasikleri](https://huggingface.co/datasets/gururaser/ithaki-bilimkurgu-klasikleri)

---

## 📈 Türkçe MMLU Benchmark Test Sonuçları

Model, [alibayram/yapay_zeka_turkce_mmlu_bolum_sonuclari](https://huggingface.co/datasets/alibayram/yapay_zeka_turkce_mmlu_bolum_sonuclari) test veri kümesi kullanılarak Türkçe MMLU benchmark değerlendirmesine tabi tutulmuştur.

{markdown_table}

### 🧪 Test Metodolojisi ve Detaylar
- **Değerlendirme Scripti:** `alibayram/yapay_zeka_turkce_mmlu_bolum_sonuclari/olcum.py` mantığı temel alınmıştır.
- **Anlamsal Karşılaştırma:** `paraphrase-multilingual-mpnet-base-v2` modeli ile şık eşleştirme ve opsiyonel harf eşleme.
- **Ortam:** Google Colab T4 GPU (4-bit NF4 Quantization)
- **Test Seti:** alibayram/yapay_zeka_turkce_mmlu_model_cevaplari

["<img src=\"https://raw.githubusercontent.com/unslothai/unsloth/main/images/unsloth%20made%20with%20love.png\" width=\"200\"/>"](https://github.com/unslothai/unsloth)
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=guncel_readme.encode('utf-8'),
    path_in_repo='README.md',
    repo_id=repo_id,
    repo_type='model'
)
print(f"\n✅ Model Kartı (README.md) Başarıyla Güncellendi!")
print(f"🔗 Model Linki: https://huggingface.co/{repo_id}")


✅ Model Kartı (README.md) Başarıyla Güncellendi!
🔗 Model Linki: https://huggingface.co/gururaser/qwen3-4b-bilimkurgu
